# AI Engineering Jobs Analysis

Analysis of 8,051 job descriptions extracted from builtin.com across nine monthly snapshots from February 4 through September 23, 2026.

Search results cover LA (Global), New York, London, Amsterdam, Berlin, and India. This notebook summarizes the combined structured dataset.

In [1]:
import yaml
import pandas as pd
from pathlib import Path
from collections import Counter

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', 80)

In [2]:
# Load all structured YAML files into a flat DataFrame
STRUCTURED_DIR = Path('data_structured')

records = []
for file in STRUCTURED_DIR.rglob('*.yaml'):
    try:
        with open(file, 'r', encoding='utf-8') as f:
            job = yaml.safe_load(f)

        pos = job.get('position', {})
        comp = job.get('company', {})
        skills = pos.get('skills', {})
        records.append({
            'job_id': job.get('meta', {}).get('job_id', ''),
            'company': comp.get('name', ''),
            'stage': comp.get('stage', ''),
            'focus': comp.get('focus', ''),
            'title': pos.get('title', ''),
            'ai_type': pos.get('ai_type', {}).get('type', 'unknown'),
            'is_customer_facing': pos.get('is_customer_facing', False),
            'is_management': pos.get('is_management', False),
            'responsibilities': pos.get('responsibilities', []),
            'use_cases': comp.get('use_cases', pos.get('use_cases', [])),
            'skills_genai': skills.get('genai', []),
            'skills_ml': skills.get('ml', []),
            'skills_web': skills.get('web', []),
            'skills_databases': skills.get('databases', []),
            'skills_data': skills.get('data', []),
            'skills_cloud': skills.get('cloud', []),
            'skills_ops': skills.get('ops', []),
            'skills_languages': skills.get('languages', []),
            'skills_domains': skills.get('domains', []),
            'skills_other': skills.get('other', []),
        })
    except Exception as e:
        print(f'Error loading {file}: {e}')

In [3]:

df = pd.DataFrame(records)

# Ensure list columns are actually lists
list_cols = [c for c in df.columns if c.startswith('skills_')] + ['responsibilities', 'use_cases']
for col in list_cols:
    df[col] = df[col].apply(lambda x: x if isinstance(x, list) else [])

print(f'Loaded {len(df)} jobs')
df.head(3)

Loaded 8051 jobs


,job_id,company,stage,focus,title,ai_type,is_customer_facing,is_management,responsibilities,use_cases,skills_genai,skills_ml,skills_web,skills_databases,skills_data,skills_cloud,skills_ops,skills_languages,skills_domains,skills_other
0,10444475,The Trade Desk,NaN,Independent programmatic digital advertising platform,Sr AI Enablement Engineer,ai-first,False,False,[Design and deploy intelligent agentic systems integrating LLMs with enterpr...,[Internal AI solutions for productivity and workflow streamlining across the...,"[LLMs, RAG, Agentic Workflows, LangChain, LlamaIndex, Semantic Kernel, Micro...",[],"[APIs, React]",[Vector Databases],[Data Pipelines],[Azure AI Search],[],"[Python, C#, SQL, TypeScript]",[],[]
1,10807809,CoreWeave,publicly traded,Cloud infrastructure platform for AI workloads,Senior Software Engineer - Physical AI,ai-support,False,False,[Deliver critical features and services across the full software lifecycle f...,"[Platform powering engineering simulation workflows, Foundation for agentic ...",[],[],"[APIs, OAuth]",[],"[Temporal, Airflow, Spark, Ray]","[AWS, Azure, GCP]","[CI/CD, Automated Testing, Observability, Incident Response, Distributed Sys...",[Python],[],[]
2,10374607,Bank of America,NaN,Global banking and financial services,Senior Principal Engineer (Generative AI Strategy),ai-first,False,False,[Define and lead the engineering approach and Generative AI strategy for GM ...,"[Regulatory reporting for Global Markets, Deriving insights from large struc...","[Generative AI, LLMs, Agentic Workflows]",[Machine Learning],"[Microservices, REST APIs]",[Databases],"[Data Analytics, Messaging]",[],[],[Python],[],"[Automation, Stakeholder Management, Technical Strategy Development, Archite..."


In [4]:
# Helper: explode a skill list column into counts
def skill_counts(df_subset, col):
    return df_subset[col].explode().dropna().value_counts()

# Helper: count jobs that have a specific skill (substring match) across all skill columns
def jobs_with_skill(df_subset, skill_name):
    skill_lower = skill_name.lower()
    mask = df_subset.apply(
        lambda row: any(
            skill_lower in s.lower()
            for col in SKILL_COLS
            for s in (row[col] if isinstance(row[col], list) else [])
        ), axis=1
    )
    return mask.sum()

SKILL_COLS = [c for c in df.columns if c.startswith('skills_')]

def all_skills_lower(row):
    """Get all skills from a row as lowercase strings."""
    out = []
    for col in SKILL_COLS:
        if isinstance(row[col], list):
            out.extend(s.lower() for s in row[col])
    return out

# Subsets
ai_first = df[df['ai_type'] == 'ai-first']
ai_support = df[df['ai_type'] == 'ai-support']
ml_first = df[df['ai_type'] == 'ml-first']

print(f'AI-First: {len(ai_first)}, AI-Support: {len(ai_support)}, ML: {len(ml_first)}')

AI-First: 5652, AI-Support: 1941, ML: 383


## Job Type Distribution

In [5]:
type_counts = df['ai_type'].value_counts()
type_pct = (type_counts / len(df) * 100).round(1)
pd.DataFrame({'jobs': type_counts, '%': type_pct})

,jobs,%
ai_type,,
ai-first,5652,70.2
ai-support,1941,24.1
ml-first,383,4.8
unknown,75,0.9


## Dataset Statistics

In [6]:
print(f'Unique companies: {df["company"].nunique()}')
print(f'\nTop 20 companies by job count:')
df['company'].value_counts().head(20)

Unique companies: 2772

Top 20 companies by job count:


company
Capital One                        156
Citi                               109
Optum                              106
NVIDIA                              73
BJAK                                60
Thomson Reuters                     53
Hewlett Packard Enterprise          51
Wells Fargo                         50
JPMorganChase                       48
Wolters Kluwer                      41
PwC                                 40
Microsoft                           39
NextHire Consulting                 39
Jack & Jill AI                      39
G2i                                 38
Ecolab                              37
OpenAI                              33
Vrinda International                33
Cisco                               32
New York Life Insurance Company     32
Name: count, dtype: int64

In [7]:
stage_counts = df[df['stage'] != '']['stage'].value_counts()
stage_pct = (stage_counts / len(df) * 100).round(1)
pd.DataFrame({'jobs': stage_counts, '%': stage_pct})

,jobs,%
stage,,
Series A,53,0.7
Series B,53,0.7
Series C,41,0.5
startup,28,0.3
Publicly traded (NASDAQ: NICE),23,0.3
...,...,...
"Publicly traded (NYSE: IOT, recently public)",1,0.0
Bootstrapped (founded with no investment capital),1,0.0
"Raised $18M+ led by Navitas Capital, Resolute Ventures and Y Combinator",1,0.0


In [8]:
print(f'Customer-facing roles: {df["is_customer_facing"].sum()} ({df["is_customer_facing"].mean()*100:.1f}%)')
print(f'Management roles: {df["is_management"].sum()} ({df["is_management"].mean()*100:.1f}%)')
print(f'\nMost common job titles:')
df['title'].value_counts().head(10)

Customer-facing roles: 1364 (16.9%)
Management roles: 202 (2.5%)

Most common job titles:


title
AI Engineer                   386
Senior AI Engineer            244
Applied AI Engineer            88
AI/ML Engineer                 86
Lead AI Engineer               80
Senior AI/ML Engineer          60
Staff AI Engineer              58
Principal AI Engineer          56
AI Product Engineer            33
Senior Applied AI Engineer     33
Name: count, dtype: int64

## Skills Analysis

In [9]:
n = len(df)
genai = skill_counts(df, 'skills_genai')
print('Top GenAI skills:')
pd.DataFrame({'jobs': genai.head(10), '%': (genai.head(10) / n * 100).round(1)})

Top GenAI skills:


,jobs,%
skills_genai,,
LLMs,5000,62.1
AI Agents,3323,41.3
RAG,3264,40.5
Prompt Engineering,2808,34.9
Agentic Workflows,2571,31.9
Generative AI,2357,29.3
LLM Evaluation,1784,22.2
LangChain,1776,22.1
Embeddings,1334,16.6


In [10]:
print('Top ML skills:')
skill_counts(df, 'skills_ml').head(10)

Top ML skills:


skills_ml
Machine Learning    2990
PyTorch             1450
TensorFlow          1054
Model Evaluation     969
Deep Learning        725
scikit-learn         607
Model Training       570
Hugging Face         564
Transformers         438
NLP                  378
Name: count, dtype: int64

In [11]:
print('Top web skills:')
skill_counts(df, 'skills_web').head(10)

Top web skills:


skills_web
APIs                      2129
REST APIs                 1332
React                     1193
Microservices             1186
FastAPI                    786
Full-Stack Development     492
Flask                      315
Angular                    284
Next.js                    268
GraphQL                    243
Name: count, dtype: int64

In [12]:
print('Top database skills:')
skill_counts(df, 'skills_databases').head(10)

Top database skills:


skills_databases
Vector Databases    1886
PostgreSQL           880
Pinecone             590
NoSQL                560
Vector Search        537
Snowflake            477
Weaviate             408
SQL                  403
MongoDB              318
Redis                312
Name: count, dtype: int64

In [13]:
print('Top cloud skills:')
skill_counts(df, 'skills_cloud').head(5)

Top cloud skills:


skills_cloud
AWS            3247
Azure          2424
GCP            2209
AWS Bedrock     482
Vertex AI       423
Name: count, dtype: int64

In [14]:
print('Top ops skills:')
skill_counts(df, 'skills_ops').head(10)

Top ops skills:


skills_ops
CI/CD                  2976
Docker                 1967
Kubernetes             1934
Observability          1743
MLOps                  1370
Distributed Systems    1179
Monitoring             1149
Model Deployment       1088
Git                     928
Terraform               790
Name: count, dtype: int64

In [15]:
langs = skill_counts(df, 'skills_languages')
print('Top languages:')
pd.DataFrame({'jobs': langs.head(10), '%': (langs.head(10) / n * 100).round(1)})

Top languages:


,jobs,%
skills_languages,,
Python,5698,70.8
TypeScript,1555,19.3
Java,1420,17.6
SQL,1174,14.6
Go,927,11.5
JavaScript,924,11.5
C++,600,7.5
C#,530,6.6
Node.js,499,6.2


## GenAI Framework Ecosystem

In [16]:
frameworks = ['LangChain', 'LangGraph', 'LlamaIndex', 'CrewAI', 'AutoGen']
genai_all = skill_counts(df, 'skills_genai')
fw = genai_all[genai_all.index.isin(frameworks)].reindex(frameworks).dropna().astype(int)
pd.DataFrame({'jobs': fw, '%': (fw / n * 100).round(1)})

,jobs,%
skills_genai,,
LangChain,1776,22.1
LangGraph,1141,14.2
LlamaIndex,666,8.3
CrewAI,543,6.7
AutoGen,436,5.4


## Supporting Roles: What AI-Support Engineers Do

In [17]:
def categorize_support_role(row):
    title = row['title'].lower()
    resp = ' '.join(row['responsibilities']).lower()
    categories = {
        'Platform/Infrastructure': ['platform', 'infrastructure', 'infra', 'mlops', 'kubernetes', 'k8s', 'deployment'],
        'Data/Pipelines': ['data engineer', 'data pipeline', 'etl', 'data platform'],
        'Sales/Solutions': ['sales', 'solutions', 'presales', 'customer success'],
        'Backend/General SWE': ['backend', 'api', 'microservices', 'internal tools'],
        'Frontend/UI': ['frontend', 'ui', 'ux', 'full-stack'],
    }
    for cat, keywords in categories.items():
        if any(kw in title or kw in resp for kw in keywords):
            return cat
    return 'Other'

support = ai_support.copy()
support['category'] = support.apply(categorize_support_role, axis=1)

print(f'{len(ai_support)} jobs ({len(ai_support)/len(df)*100:.1f}%) classified as AI-Support\n')
support['category'].value_counts()

1941 jobs (24.1%) classified as AI-Support



category
Platform/Infrastructure    1314
Sales/Solutions             199
Backend/General SWE         177
Frontend/UI                 141
Data/Pipelines               63
Other                        47
Name: count, dtype: int64

In [18]:
# Do AI-Support roles need GenAI knowledge?
has_genai = ai_support['skills_genai'].apply(len) > 0
print(f'AI-Support roles with GenAI skills: {has_genai.sum()}/{len(ai_support)} ({has_genai.mean()*100:.1f}%)')
print(f'AI-Support roles without GenAI skills: {(~has_genai).sum()}/{len(ai_support)} ({(~has_genai).mean()*100:.1f}%)')

print('\nGenAI skills in AI-Support roles:')
support_genai = skill_counts(ai_support, 'skills_genai')
pd.DataFrame({'jobs': support_genai.head(10), '%': (support_genai.head(10) / len(ai_support) * 100).round(1)})

AI-Support roles with GenAI skills: 1143/1941 (58.9%)
AI-Support roles without GenAI skills: 798/1941 (41.1%)

GenAI skills in AI-Support roles:


,jobs,%
skills_genai,,
LLMs,521,26.8
Generative AI,327,16.8
AI Agents,309,15.9
GitHub Copilot,183,9.4
Claude Code,169,8.7
RAG,160,8.2
Prompt Engineering,156,8.0
Cursor,151,7.8
Anthropic API,131,6.7


### Skill Comparison: AI-First vs AI-Support

In [19]:
compare_skills = ['RAG', 'prompt engineering', 'agents', 'LangChain', 'Docker', 'Kubernetes', 'AWS', 'React']

rows = []
for skill in compare_skills:
    af = jobs_with_skill(ai_first, skill)
    asp = jobs_with_skill(ai_support, skill)
    rows.append({
        'skill': skill,
        'AI-First': f'{af/len(ai_first)*100:.1f}%',
        'AI-Support': f'{asp/len(ai_support)*100:.1f}%',
    })

pd.DataFrame(rows).set_index('skill')

,AI-First,AI-Support
skill,,
RAG,55.6%,10.5%
prompt engineering,46.8%,8.0%
agents,54.1%,16.7%
LangChain,29.8%,4.5%
Docker,24.8%,25.3%
Kubernetes,22.5%,30.6%
AWS,45.6%,39.8%
React,15.6%,18.1%


## Research vs Applied Roles

In [20]:
def is_research_role(row):
    title = row['title'].lower()
    resp = ' '.join(row['responsibilities']).lower()
    use_cases = ' '.join(row['use_cases']).lower()

    research_indicators = [
        'research', 'scientist', 'publication', 'paper', 'novel',
        'algorithm', 'architecture development', 'model architecture',
        'training methods', 'safety research', 'rl research',
        'reinforcement learning', 'world model', 'control theory',
        'experimental', 'push sota', 'state of the art'
    ]
    non_research_indicators = [
        'production', 'deploy', 'shipping', 'product',
        'customer', 'enterprise', 'api integration',
        'fine-tuning existing', 'apply', 'implement'
    ]

    if any(kw in title for kw in ['research engineer', 'scientist', 'research scientist']):
        return True

    all_text = f'{resp} {use_cases}'
    r_score = sum(1 for kw in research_indicators if kw in all_text)
    nr_score = sum(1 for kw in non_research_indicators if kw in all_text)
    return r_score > nr_score and r_score >= 2

df['is_research'] = df.apply(is_research_role, axis=1)
research_count = df['is_research'].sum()

pd.DataFrame([
    {'Role Type': 'Research', 'Jobs': research_count, '%': f'{research_count/len(df)*100:.1f}%'},
    {'Role Type': 'Applied/Production', 'Jobs': len(df) - research_count, '%': f'{(len(df)-research_count)/len(df)*100:.1f}%'},
]).set_index('Role Type')

,Jobs,%
Role Type,,
Research,204,2.5%
Applied/Production,7847,97.5%


In [21]:
print('Sample research titles:')
df[df['is_research']]['title'].drop_duplicates().head(15).tolist()

Sample research titles:


['AI ML Data Engineer/ Scientist',
 'Data Scientist\xa0|\xa0Data Engineer (AI & GTM Analytics)',
 'AI/ML driven ASIC Design and Implementation Principal Engineer',
 'Security Research Engineer, AI Safety and Security Engineering',
 'Lead Research Engineer - AI/ML',
 'AI Research Engineer',
 'AI Engineer Intern',
 'Senior AI Product & Research Engineer (Consultant)',
 'Lead ML & AI Engineer',
 'Data Scientist / AI & ML Engineer',
 'Senior ML Engineer (AI Research, Physical AI)',
 'Research Engineer, AI',
 'AI Engineer / Data Scientist, AI Senior Associate',
 'Senior AI Engineer / Data Scientist (Agentic AI)',
 'Senior AI Research Engineer']

## What Other Titles Do "AI Engineers" Go Under?

In [22]:
def normalize_title(title):
    t = title.lower()
    for kw in ['senior', 'staff', 'principal', 'lead', 'junior', 'sr.', 'sr', 'iii', 'ii']:
        t = t.replace(kw, '').strip()
    return ' '.join(t.split())

df['norm_title'] = df['title'].apply(normalize_title)

# Group by normalized title and ai_type
title_groups = df.groupby('norm_title')['ai_type'].value_counts().unstack(fill_value=0)
title_groups['total'] = title_groups.sum(axis=1)
title_groups = title_groups[title_groups['total'] >= 3]

# Strongly AI-First titles (75%+)
if 'ai-first' in title_groups.columns:
    title_groups['ai_first_pct'] = (title_groups['ai-first'] / title_groups['total'] * 100).round(0)
    strongly_ai_first = title_groups[title_groups['ai_first_pct'] >= 75].sort_values('total', ascending=False)
    print('Strongly AI-First titles (75%+ AI-First):')
    print(strongly_ai_first[['total', 'ai_first_pct']].head(10).to_string())

# Strongly AI-Support titles (75%+)
if 'ai-support' in title_groups.columns:
    title_groups['ai_support_pct'] = (title_groups['ai-support'] / title_groups['total'] * 100).round(0)
    strongly_support = title_groups[title_groups['ai_support_pct'] >= 75].sort_values('total', ascending=False)
    print('\nStrongly AI-Support titles (75%+ AI-Support):')
    print(strongly_support[['total', 'ai_support_pct']].head(10).to_string())

Strongly AI-First titles (75%+ AI-First):
ai_type                 total  ai_first_pct
norm_title                                 
ai engineer               912          92.0
ai/ml engineer            201          76.0
applied ai engineer       137          93.0
ai software engineer       67          79.0
ai developer               60          80.0
ai product engineer        46          83.0
software engineer - ai     44          82.0
ai solutions engineer      40          88.0
software engineer, ai      40          92.0
agentic ai engineer        37         100.0

Strongly AI-Support titles (75%+ AI-Support):
ai_type                                               total  ai_support_pct
norm_title                                                                 
ai infrastructure engineer                               16            81.0
product engineer, ai                                      7            86.0
ai devops engineer                                        6            83.0
ai 

## How Much ML Do AI Engineers Need to Know?

In [23]:
ml_skills_list = [
    'PyTorch', 'TensorFlow', 'Keras', 'JAX', 'scikit-learn', 'XGBoost',
    'LightGBM', 'fine-tuning', 'model training', 'model evaluation',
    'embeddings', 'deep learning', 'machine learning', 'neural networks',
    'optimization', 'CUDA', 'transformers', 'huggingface'
]

def has_any_ml_skill(row):
    skills = all_skills_lower(row)
    return any(ml.lower() in s for s in skills for ml in ml_skills_list)

ai_first_ml = ai_first.apply(has_any_ml_skill, axis=1)
print(f'{ai_first_ml.mean()*100:.1f}% of AI-First roles require some ML knowledge')

# Most common ML skills in AI-First roles
def count_ml_skill(skill_name):
    skill_lower = skill_name.lower()
    return ai_first.apply(
        lambda row: any(skill_lower in s for s in all_skills_lower(row)), axis=1
    ).sum()

ml_counts = {s: count_ml_skill(s) for s in ml_skills_list}
ml_df = pd.Series(ml_counts).sort_values(ascending=False)
ml_df = ml_df[ml_df > 0]
pd.DataFrame({'jobs': ml_df, '%': (ml_df / len(ai_first) * 100).round(1)}).head(10)

64.3% of AI-First roles require some ML knowledge


,jobs,%
machine learning,2256,39.9
fine-tuning,1401,24.8
embeddings,1297,22.9
PyTorch,1096,19.4
model evaluation,835,14.8
TensorFlow,766,13.6
model training,549,9.7
deep learning,513,9.1
scikit-learn,446,7.9
transformers,380,6.7


## What Else (Besides GenAI) Do AI Engineers Need?

In [24]:
n_af = len(ai_first)

has_genai_col = ai_first['skills_genai'].apply(len) > 0
has_ml = ai_first['skills_ml'].apply(len) > 0
has_web = ai_first['skills_web'].apply(len) > 0
has_ops = ai_first['skills_ops'].apply(len) > 0
has_cloud = ai_first['skills_cloud'].apply(len) > 0
has_data = ai_first['skills_data'].apply(len) > 0
has_db = ai_first['skills_databases'].apply(len) > 0
has_any_other = has_ml | has_web | has_ops | has_cloud | has_data | has_db

combos = {
    'GenAI + Ops (Docker, K8s, CI/CD)': (has_genai_col & has_ops).sum(),
    'GenAI + ML skills': (has_genai_col & has_ml).sum(),
    'GenAI + Web skills': (has_genai_col & has_web).sum(),
    'GenAI + ANY other tech': (has_genai_col & has_any_other).sum(),
    'Pure GenAI (nothing else)': (has_genai_col & ~has_any_other).sum(),
}

combo_df = pd.Series(combos)
pd.DataFrame({'jobs': combo_df, '%': (combo_df / n_af * 100).round(1)})

,jobs,%
"GenAI + Ops (Docker, K8s, CI/CD)",4274,75.6
GenAI + ML skills,3116,55.1
GenAI + Web skills,3502,62.0
GenAI + ANY other tech,5357,94.8
Pure GenAI (nothing else),216,3.8


In [25]:
# Non-GenAI skills by category for AI-First roles
for cat in ['web', 'cloud', 'ops', 'languages', 'databases', 'data']:
    col = f'skills_{cat}'
    counts = skill_counts(ai_first, col)
    if len(counts) > 0:
        top = counts.head(6)
        pcts = (top / n_af * 100).round(1)
        print(f'\n{cat.upper()}:')
        for skill, count in top.items():
            print(f'  {skill}: {count} ({pcts[skill]}%)')


WEB:
  APIs: 1685 (29.8%)
  REST APIs: 937 (16.6%)
  Microservices: 884 (15.6%)
  React: 828 (14.6%)
  FastAPI: 674 (11.9%)
  Full-Stack Development: 375 (6.6%)

CLOUD:
  AWS: 2359 (41.7%)
  Azure: 1747 (30.9%)
  GCP: 1621 (28.7%)
  AWS Bedrock: 436 (7.7%)
  Vertex AI: 357 (6.3%)
  AWS Lambda: 302 (5.3%)

OPS:
  CI/CD: 2019 (35.7%)
  Docker: 1396 (24.7%)
  Observability: 1342 (23.7%)
  Kubernetes: 1267 (22.4%)
  MLOps: 1070 (18.9%)
  Monitoring: 831 (14.7%)

LANGUAGES:
  Python: 4287 (75.8%)
  TypeScript: 1171 (20.7%)
  Java: 932 (16.5%)
  SQL: 791 (14.0%)
  Go: 650 (11.5%)
  JavaScript: 603 (10.7%)

DATABASES:
  Vector Databases: 1774 (31.4%)
  PostgreSQL: 656 (11.6%)
  Pinecone: 556 (9.8%)
  Vector Search: 508 (9.0%)
  Weaviate: 389 (6.9%)
  NoSQL: 344 (6.1%)

DATA:
  Data Pipelines: 1223 (21.6%)
  Databricks: 381 (6.7%)
  Kafka: 361 (6.4%)
  Spark: 279 (4.9%)
  Data Modeling: 259 (4.6%)
  Airflow: 239 (4.2%)


In [26]:
# Full-stack expectations for AI-First roles
def has_frontend_skills(row):
    skills = all_skills_lower(row)
    return any(kw in s for s in skills for kw in ['react', 'vue', 'next.js', 'frontend', 'typescript', 'javascript'])

def has_backend_skills(row):
    skills = all_skills_lower(row)
    return any(kw in s for s in skills for kw in ['fastapi', 'flask', 'django', 'api', 'graphql', 'rest'])

fe = ai_first.apply(has_frontend_skills, axis=1)
be = ai_first.apply(has_backend_skills, axis=1)
fs = fe & be

print(f'Frontend skills: {fe.sum()}/{n_af} ({fe.mean()*100:.1f}%)')
print(f'Backend skills: {be.sum()}/{n_af} ({be.mean()*100:.1f}%)')
print(f'Full-stack (both): {fs.sum()}/{n_af} ({fs.mean()*100:.1f}%)')

Frontend skills: 1737/5652 (30.7%)
Backend skills: 3418/5652 (60.5%)
Full-stack (both): 1295/5652 (22.9%)


## Fine-Tuning Requirements

In [27]:
ft_keywords = ['fine-tun', 'finetun', 'fine tun', 'custom model', 'specialized model',
               'domain-specific', 'adaptation', 'lora', 'qlora', 'peft', 'instruction tuning']

def get_all_text(row):
    parts = [row['title'], ' '.join(row['responsibilities']), ' '.join(row['use_cases'])]
    for col in SKILL_COLS:
        if isinstance(row[col], list):
            parts.extend(row[col])
    return ' '.join(parts).lower()

ai_first_texts = ai_first.apply(get_all_text, axis=1)
has_ft = ai_first_texts.apply(lambda t: any(kw in t for kw in ft_keywords))

print(f'{has_ft.mean()*100:.1f}% of AI-First roles mention fine-tuning')

# Depth of fine-tuning
primary_ft_kw = ['lora', 'qlora', 'peft']

def ft_depth(text):
    if not any(kw in text for kw in ft_keywords):
        return 'No FT mentioned'
    if any(kw in text for kw in primary_ft_kw) or text.count('fine-tun') + text.count('finetun') >= 2:
        return 'Primary FT responsibility'
    return 'Secondary/occasional FT'

ft_levels = ai_first_texts.apply(ft_depth).value_counts()
pd.DataFrame({'jobs': ft_levels, '%': (ft_levels / len(ai_first) * 100).round(1)})

27.7% of AI-First roles mention fine-tuning


,jobs,%
No FT mentioned,4085,72.3
Primary FT responsibility,951,16.8
Secondary/occasional FT,616,10.9


In [28]:
# Fine-tuning use cases
ft_use_case_categories = {
    'Instruction following': ['instruction', 'task', 'command', 'reasoning', 'agent'],
    'Domain knowledge': ['domain', 'industry', 'vertical', 'medical', 'legal', 'finance', 'healthcare', 'scientific'],
    'Style/Tone': ['style', 'tone', 'voice', 'brand', 'personality', 'format'],
    'Company data': ['company', 'internal', 'proprietary', 'organization'],
    'Performance': ['faster', 'smaller', 'efficiency', 'latency', 'cost', 'optimize'],
    'Language': ['language', 'translation', 'multilingual', 'non-english'],
    'Privacy': ['privacy', 'on-premise', 'local', 'offline', 'secure'],
}

ft_jobs = ai_first[has_ft.values]
all_ucs = ft_jobs['use_cases'].explode().dropna()

uc_cats = Counter()
for uc in all_ucs:
    uc_lower = uc.lower()
    for cat, kws in ft_use_case_categories.items():
        if any(kw in uc_lower for kw in kws):
            uc_cats[cat] += 1
            break

print('Fine-tuning use cases:')
for cat, count in sorted(uc_cats.items(), key=lambda x: -x[1]):
    print(f'  {cat}: {count}')

Fine-tuning use cases:
  Instruction following: 1471
  Domain knowledge: 514
  Company data: 249
  Style/Tone: 221
  Performance: 211
  Language: 106
  Privacy: 75


## Evaluation and Production: How Important Are They?

In [29]:
# How often do responsibilities mention evaluation/quality vs production/deployment?
all_resp = df['responsibilities'].explode().dropna()
print(f'Total responsibilities: {len(all_resp)}')

eval_keywords = ['evaluat', 'quality', 'test', 'monitor', 'observ', 'metric', 'hallucinat', 'guardrail', 'bias', 'safety']
prod_keywords = ['deploy', 'production', 'ship', 'release', 'scale', 'reliab', 'latency', 'infra', 'ci/cd', 'docker', 'kubernetes']

def count_resp_keyword(responsibilities, keywords):
    return sum(1 for r in responsibilities if any(kw in r.lower() for kw in keywords))

eval_count = count_resp_keyword(all_resp, eval_keywords)
prod_count = count_resp_keyword(all_resp, prod_keywords)

print(f'\nResponsibilities mentioning evaluation/quality: {eval_count} ({eval_count/len(all_resp)*100:.1f}%)')
print(f'Responsibilities mentioning production/deployment: {prod_count} ({prod_count/len(all_resp)*100:.1f}%)')

# How many JOBS mention evaluation in responsibilities?
def job_mentions(df, keywords):
    return df['responsibilities'].apply(
        lambda resps: any(any(kw in r.lower() for kw in keywords) for r in resps)
    ).sum()

eval_jobs = job_mentions(df, eval_keywords)
prod_jobs = job_mentions(df, prod_keywords)

print(f'\nJobs with evaluation/quality responsibilities: {eval_jobs}/{len(df)} ({eval_jobs/len(df)*100:.1f}%)')
print(f'Jobs with production/deployment responsibilities: {prod_jobs}/{len(df)} ({prod_jobs/len(df)*100:.1f}%)')

Total responsibilities: 58060



Responsibilities mentioning evaluation/quality: 13126 (22.6%)
Responsibilities mentioning production/deployment: 17406 (30.0%)

Jobs with evaluation/quality responsibilities: 6483/8051 (80.5%)
Jobs with production/deployment responsibilities: 7064/8051 (87.7%)


In [30]:
# Most common action words in responsibilities
import re

action_words = [
    'build', 'design', 'implement', 'develop', 'deploy', 'maintain',
    'collaborate', 'monitor', 'optimize', 'evaluate', 'test', 'scale',
    'integrate', 'architect', 'manage', 'create', 'lead', 'research',
    'support', 'deliver', 'ship', 'automate', 'improve', 'ensure'
]

word_counts = Counter()
for resp in all_resp:
    resp_lower = resp.lower()
    for word in action_words:
        if re.search(r'\b' + word + r'\w*\b', resp_lower):
            word_counts[word] += 1

action_df = pd.Series(dict(word_counts.most_common()))
pd.DataFrame({
    'mentions': action_df,
    '% of responsibilities': (action_df / len(all_resp) * 100).round(1)
})

,mentions,% of responsibilities
design,9937,17.1
build,9585,16.5
develop,8582,14.8
deploy,5276,9.1
architect,5016,8.6
implement,4960,8.5
test,3806,6.6
maintain,3498,6.0
lead,3251,5.6
manage,3043,5.2


In [31]:
# Evaluation and production/ops as explicit skill requirements
eval_skill_keywords = ['evaluation', 'eval', 'testing', 'quality', 'monitoring', 'observability']
prod_skill_keywords = ['docker', 'kubernetes', 'ci/cd', 'mlops', 'terraform']

def count_jobs_with_skills(df_subset, keywords):
    return df_subset.apply(
        lambda row: any(
            any(kw in s.lower() for kw in keywords)
            for col in SKILL_COLS
            for s in (row[col] if isinstance(row[col], list) else [])
        ), axis=1
    ).sum()

eval_skill_jobs = count_jobs_with_skills(df, eval_skill_keywords)
prod_skill_jobs = count_jobs_with_skills(df, prod_skill_keywords)

print(f'Jobs with evaluation-related skills: {eval_skill_jobs}/{len(df)} ({eval_skill_jobs/len(df)*100:.1f}%)')
print(f'Jobs with production/ops skills: {prod_skill_jobs}/{len(df)} ({prod_skill_jobs/len(df)*100:.1f}%)')

# For AI-First specifically
eval_af = count_jobs_with_skills(ai_first, eval_skill_keywords)
prod_af = count_jobs_with_skills(ai_first, prod_skill_keywords)

print(f'\nAI-First jobs with evaluation skills: {eval_af}/{len(ai_first)} ({eval_af/len(ai_first)*100:.1f}%)')
print(f'AI-First jobs with production/ops skills: {prod_af}/{len(ai_first)} ({prod_af/len(ai_first)*100:.1f}%)')

Jobs with evaluation-related skills: 4367/8051 (54.2%)
Jobs with production/ops skills: 4315/8051 (53.6%)



AI-First jobs with evaluation skills: 3442/5652 (60.9%)
AI-First jobs with production/ops skills: 3012/5652 (53.3%)
